In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [4]:
# The tutorial uses pandas to read the CSV file and use the information from the dataframe to come up with a schema

import pandas as pd

In [ ]:
# This throws an error. However, it is not  required as the schema is defined for us below

# df_green_pd = pd.read_csv('data/raw/green/2020/01/green_tripdata_2020_01.csv.gz', nrows=100)

In [ ]:
# Note this also creates an error. Not really required

# spark.createDataFrame(df_green_pd.pd)

In [30]:
from pyspark.sql import types

In [31]:
green_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("ehail_fee", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("trip_type", types.IntegerType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [32]:
# Reads the CSV file from the input_path, applying the green_schema and writes the transformed data to Parquet format in output_path
# Note: this works. It only throws an error on 2021/08 because of empty data. 
years = [2020,2021]

for year in years:
    for month in range(1, 13):
        print(f'processing data for {year}/{month}')
    
        input_path = f'data/raw/green/{year}/{month:02d}/'
        output_path = f'data/pq/green/{year}/{month:02d}/'
    
        df_green = spark.read \
            .option("header", "true") \
            .schema(green_schema) \
            .csv(input_path)
    
        df_green \
            .repartition(4) \
            .write.parquet(output_path)

processing data for 2020/1
processing data for 2020/2
processing data for 2020/3
processing data for 2020/4
processing data for 2020/5
processing data for 2020/6
processing data for 2020/7
processing data for 2020/8
processing data for 2020/9
processing data for 2020/10
processing data for 2020/11
processing data for 2020/12
processing data for 2021/1
processing data for 2021/2
processing data for 2021/3
processing data for 2021/4
processing data for 2021/5
processing data for 2021/6
processing data for 2021/7
processing data for 2021/8


Py4JJavaError: An error occurred while calling o1127.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 151.0 failed 1 times, most recent failure: Lost task 0.0 in stage 151.0 (TID 301) (Socala executor driver): java.io.EOFException: Unexpected end of input stream
	at org.apache.hadoop.io.compress.DecompressorStream.decompress(DecompressorStream.java:145)
	at org.apache.hadoop.io.compress.DecompressorStream.read(DecompressorStream.java:85)
	at java.base/java.io.InputStream.read(InputStream.java:205)
	at org.apache.hadoop.util.LineReader.fillBuffer(LineReader.java:180)
	at org.apache.hadoop.util.LineReader.readDefaultLine(LineReader.java:216)
	at org.apache.hadoop.util.LineReader.readLine(LineReader.java:174)
	at org.apache.hadoop.mapreduce.lib.input.LineRecordReader.skipUtfByteOrderMark(LineRecordReader.java:144)
	at org.apache.hadoop.mapreduce.lib.input.LineRecordReader.nextKeyValue(LineRecordReader.java:184)
	at org.apache.spark.sql

In [33]:
#Yellow
# Note: this works. It only throws an error on 2021/08 because of empty data. 
# Note: Not sure the repartitions worked. To be checked.

years = [2020,2021]

for year in years:
    for month in range(1, 13):
        print(f'processing data for {year}/{month}')

        input_path = f'data/raw/yellow/{year}/{month:02d}/'
        output_path = f'data/pq/yellow/{year}/{month:02d}/'

        df_yellow = spark.read.option("header", "true").schema(yellow_schema).csv(input_path)

        df_yellow.repartition(4).write.parquet(output_path)

processing data for 2020/1
processing data for 2020/2
processing data for 2020/3
processing data for 2020/4
processing data for 2020/5
processing data for 2020/6
processing data for 2020/7
processing data for 2020/8
processing data for 2020/9
processing data for 2020/10
processing data for 2020/11
processing data for 2020/12
processing data for 2021/1
processing data for 2021/2
processing data for 2021/3
processing data for 2021/4
processing data for 2021/5
processing data for 2021/6
processing data for 2021/7
processing data for 2021/8


Py4JJavaError: An error occurred while calling o1409.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 190.0 failed 1 times, most recent failure: Lost task 0.0 in stage 190.0 (TID 378) (Socala executor driver): java.io.EOFException: Unexpected end of input stream
	at org.apache.hadoop.io.compress.DecompressorStream.decompress(DecompressorStream.java:145)
	at org.apache.hadoop.io.compress.DecompressorStream.read(DecompressorStream.java:85)
	at java.base/java.io.InputStream.read(InputStream.java:205)
	at org.apache.hadoop.util.LineReader.fillBuffer(LineReader.java:180)
	at org.apache.hadoop.util.LineReader.readDefaultLine(LineReader.java:216)
	at org.apache.hadoop.util.LineReader.readLine(LineReader.java:174)
	at org.apache.hadoop.mapreduce.lib.input.LineRecordReader.skipUtfByteOrderMark(LineRecordReader.java:144)
	at org.apache.hadoop.mapreduce.lib.input.LineRecordReader.nextKeyValue(LineRecordReader.java:184)
	at org.apache.spark.sql

In [38]:
# checking files 
!ls -lh data/pq/green/2020/

total 48K
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 01
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 02
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 03
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 04
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 05
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 06
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 07
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 08
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 09
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 10
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 11
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 12


In [39]:
# Reading parquet with pyspark
# Not sure why it did not read the data. Unless this has to be done in a new PysSpark shell as shown in notes of Manuel G.
df_green = spark.read.parquet('data/pq/green/*/*')

In [41]:
# Not sure why it did not read the data.
# Unless this has to be done in a new PysSpark shell as shown in notes of Manuel G.
df_green.head(5)

[]

In [42]:
df_green \
    .select('lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_distance') \
    .show(5)

+--------------------+---------------------+------------+------------+-------------+
|lpep_pickup_datetime|lpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|
+--------------------+---------------------+------------+------------+-------------+
+--------------------+---------------------+------------+------------+-------------+

